# RFSoC 4x2 OOK 圖片迴路

使用本發布包的 `ook_rf.bit`、`ook_rf.hwh`、`manifest.json` 和 `run_on_board.py`。將整包解壓到板上的同一資料夾，從 PYNQ Jupyter 開啟這份 Notebook。接線為 **DAC_A（DAC0）→ 50 Ω 同軸線與 10 dB 衰減器 → ADC_A（ADC0）**。

流程參考助教的 `OnlyOneBoard.ipynb`，先初始化，再驗證時脈與 RF 路徑，最後顯示收發結果。第一次請逐格執行，看到 `OK` 再進下一格。若前一步出錯，先查看 `received/bringup.log` 和 `received/result.json`。

## 1. 選擇圖片並檢查檔案

預設使用包內的 640 × 480 灰階圖。更換 PNG 或 JPG 時，程式會轉成同樣大小的灰階圖。這一格會核對 bitstream 與 HWH 的雜湊，並透過 `Overlay(download=False)` 解析硬體描述。板上的設定此時尚未改動。

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image
from run_on_board import BoardRun

bundle = Path.cwd()
image_path = bundle / "image.raw"
output_dir = bundle / "received"
amplitude = 4096
timeout = 30.0

if "session" in globals():
    session.close()
session = BoardRun(bundle / "ook_rf.bit", image_path, output_dir, amplitude, timeout)
session.inspect()

## 2. 設定板上參考時脈

LMK 設為 245.76 MHz，LMX 設為 491.52 MHz。ADC 和 DAC 的 PLL 使用 491.52 MHz 參考時脈產生 4.9152 GS/s 的取樣時脈。開始這一格前，請先關閉正在操作 FPGA 的其他 Notebook kernel。[RFSoC 4x2 官方時脈初始化](https://github.com/Xilinx/RFSoC-PYNQ/blob/master/boards/RFSoC4x2/base/base.py)

In [ ]:
session.program_clocks()

## 3. 下載 OOK Overlay

用 PYNQ 載入已核對的 `ook_rf.bit`。程式會在開始與完成時各記一行，方便辨認下載是否完成。[PYNQ Overlay 文件](https://pynq.readthedocs.io/en/v3.1/pynq_package/pynq.overlay.html)

In [ ]:
session.download_overlay()

## 4. 確認 PS 可以讀取 FPGA 狀態

先讀由 PS 100 MHz 時脈驅動的 `host_state` GPIO，確認 AXI 介面有回應。這一步只檢查 PS 端。若顯示 `RF clock ready: False`，下一格會等待 RF 時脈和轉換器就緒。[PYNQ AxiGPIO 文件](https://pynq.readthedocs.io/en/v3.1/pynq_libraries/axigpio.html)

In [ ]:
session.check_ps_bus()

## 5. 確認 RFDC 就緒

等 ADC_A、DAC_A 的 tile 完成上電，確認兩個 PLL 鎖定且 RF 時脈有效。接著讀回 ADC −500 MHz、DAC +500 MHz 的 mixer 設定，確認 FIFO 已啟用。若等不到就緒，程式會列出各 tile 的狀態。[AMD RFDC 手冊](https://docs.amd.com/r/en-US/pg269-rf-data-converter)

In [ ]:
session.check_rfdc()

## 6. 上傳原圖

先讓 FPGA 的 OOK 資料路徑保持重設，再把 307,200 個像素寫進原圖 BRAM 並讀回比對。圖片透過 Overlay 上的記憶體物件讀寫。

In [ ]:
session.upload()

## 7. 量測 RF 迴路

先量 DAC 靜音時的 ADC 幅度，再短暫開啟 500 MHz pilot，量測接上同軸線後的接收幅度。程式用兩組量測的中點當作 OOK 判決門檻，並關閉 pilot。

In [ ]:
session.calibrate()

## 8. 傳送 300 個封包並驗證

FPGA 收到開始命令後，將原圖編成 OOK，經 DAC_A 送出，再由 ADC_A 接回。FPGA 檢查每個封包的 CRC，遺失 ACK 時重傳。完成後，程式逐位元組比對接收圖並記錄重傳次數、耗時和 SHA-256。

In [ ]:
result = session.transfer()
result

## 9. 顯示收發圖片

左圖是送出的灰階圖，右圖是由 ADC 路徑收到的圖。圖檔和本次的驗證結果都保存在 `received/`。

In [ ]:
comparison = Image.new("L", (1280, 480))
comparison.paste(Image.frombytes("L", (640, 480), session.pixels), (0, 0))
with Image.open(output_dir / "received.png") as received:
    comparison.paste(received, (640, 0))
display(comparison)

## 提前結束

若只想測到中間某一步，可以執行下一格，讓 OOK 回到重設狀態並結束本次操作。要再測一次，從第一格重新開始。

In [ ]:
session.close()